# Meeting Audio → Summary Pipeline

This notebook runs a local pipeline to:
1. Transcribe meeting audio using a local Whisper model (via `faster-whisper`)
2. Summarize the transcript using a local OpenAI-compatible API endpoint (default: `qwen3.5:4b`)


## Prerequisites

- Python 3 in JupyterLab
- A local OpenAI-compatible API endpoint running and reachable (for example `http://127.0.0.1:8000/v1`)
- The local endpoint should expose your target chat model (default in this notebook: `qwen3.5:4b`)

The notebook installs missing Python packages and downloads Whisper model weights as needed.

In [ ]:
import importlib
import json
import subprocess
import sys
from pathlib import Path


def ensure_python_dependencies() -> None:
    """Install required Python packages if they are missing."""
    requirements = {
        "faster_whisper": "faster-whisper",
        "openai": "openai",
    }
    missing = [pkg_name for module_name, pkg_name in requirements.items() if importlib.util.find_spec(module_name) is None]

    if missing:
        print(f"Installing missing dependencies: {', '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("Python dependencies already installed.")


ensure_python_dependencies()

In [ ]:
from openai import OpenAI
from faster_whisper import WhisperModel


def create_openai_client(base_url: str, api_key: str = "local-api-key") -> OpenAI:
    """Create a client for a local OpenAI-compatible endpoint."""
    return OpenAI(base_url=base_url, api_key=api_key)


def ensure_chat_model_available(client: OpenAI, model_name: str) -> None:
    """Validate endpoint connectivity and model availability when listed by the endpoint."""
    try:
        model_list = client.models.list()
    except Exception as exc:
        raise RuntimeError("Could not connect to the configured OpenAI-compatible endpoint.") from exc

    model_ids = {model.id for model in model_list.data if getattr(model, "id", None)}
    if model_ids and model_name not in model_ids:
        available = ", ".join(sorted(model_ids))
        raise RuntimeError(f"Configured model '{model_name}' is not available. Endpoint models: {available}")


def load_whisper_model(model_size: str = "small", device: str = "cpu", compute_type: str = "int8") -> WhisperModel:
    """Load local Whisper model (downloads weights on first run)."""
    print(f"Loading Whisper model '{model_size}' on {device} ({compute_type})...")
    return WhisperModel(model_size, device=device, compute_type=compute_type)


def transcribe_audio(audio_path: str, stt_model: WhisperModel) -> str:
    """Transcribe an audio file into plain text."""
    audio_file = Path(audio_path).expanduser().resolve()
    if not audio_file.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_file}")

    segments, info = stt_model.transcribe(str(audio_file), beam_size=5, vad_filter=True)
    transcript = " ".join(segment.text.strip() for segment in segments if segment.text.strip())

    if not transcript:
        raise RuntimeError("No speech content could be transcribed from the audio file.")

    print(f"Transcription complete. Detected language: {info.language} (prob={info.language_probability:.2f})")
    return transcript


def summarize_transcript(
    transcript: str,
    client: OpenAI,
    llm_model: str = "qwen3.5:4b",
    timeout_seconds: int = 600,
) -> str:
    """Generate a meeting summary with action items via a local OpenAI-compatible endpoint.

    Increase `timeout_seconds` for longer transcripts or slower local hardware.
    """
    prompt = f"""You are an assistant that summarizes meeting transcripts.
Return:
1) A concise summary (5-10 bullet points)
2) Key decisions
3) Action items with owners if mentioned
4) Open questions

Transcript:
{transcript}
"""

    response = client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "You summarize meeting transcripts."},
            {"role": "user", "content": prompt},
        ],
        timeout=timeout_seconds,
    )

    summary = (response.choices[0].message.content or "").strip()
    if not summary:
        raise RuntimeError("LLM returned an empty summary.")
    return summary


def summarize_meeting_audio(
    audio_path: str,
    openai_base_url: str,
    openai_api_key: str = "local-api-key",
    whisper_model_size: str = "small",
    llm_model: str = "qwen3.5:4b",
    timeout_seconds: int = 600,
) -> dict:
    """End-to-end orchestration: audio -> transcript -> summary."""
    client = create_openai_client(base_url=openai_base_url, api_key=openai_api_key)
    ensure_chat_model_available(client, llm_model)
    stt_model = load_whisper_model(whisper_model_size)
    transcript = transcribe_audio(audio_path, stt_model)
    summary = summarize_transcript(
        transcript,
        client=client,
        llm_model=llm_model,
        timeout_seconds=timeout_seconds,
    )

    return {
        "audio_path": str(Path(audio_path).expanduser().resolve()),
        "transcript": transcript,
        "summary": summary,
    }

In [ ]:
# Configure these values for your local environment, then run this cell.
AUDIO_FILE_PATH = "/absolute/path/to/meeting_audio.wav"
OPENAI_BASE_URL = "http://127.0.0.1:8000/v1"
OPENAI_API_KEY = "local-api-key"

result = summarize_meeting_audio(
    audio_path=AUDIO_FILE_PATH,
    openai_base_url=OPENAI_BASE_URL,
    openai_api_key=OPENAI_API_KEY,
    whisper_model_size="small",
    llm_model="qwen3.5:4b",
)

print("=== SUMMARY ===\n")
print(result["summary"])

# Optionally inspect the transcript:
# print(result["transcript"])

In [ ]:
# Optional: save output to JSON
output_path = Path("meeting_summary_output.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Saved output to: {output_path.resolve()}")